# Person 1 — Data collection and source inventory

Part of the six-person Basil Leaf ML pipeline. Run the numbered notebooks in order. This notebook states its inputs, produces a concrete handoff in `parts/artifacts`, and does not overwrite the complete project's `outputs/` results.

## Responsibility
Verify the source and licence information, locate the downloaded folders, count images, identify missing coverage, and create the inventory handed to Person 2. This person must not invent missing files or infer labels from regional folder names.

**Input:** `data/raw/`  
**Output:** `parts/artifacts/01_inventory.csv` and `01_collection_report.json`

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "Basil_Leaf_ML_Workflow.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project folder or parts folder.")
DATA_DIR = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "parts" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SEED = 42
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
FOLDERS = {
    "Amravati_Region_Basil_Plant_Healthy": ("Healthy", 31),
    "Nagpur_Region_Basil_Plant_Healthy": ("Healthy", 473),
    "Pune_Region_Basil_Plant_Healthy": ("Healthy", 146),
    "Basil_Plant_Unhealthy": ("Unhealthy", 481),
}
print("Project:", ROOT)
print("Python:", sys.executable)

Project: D:\SLIIT\projectr\Dataset_Train
Python: D:\SLIIT\projectr\Dataset_Train\.venv\Scripts\python.exe


In [2]:
rows=[]
for folder,(label,expected) in FOLDERS.items():
    location=DATA_DIR/folder
    files=sorted(p for p in location.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS) if location.exists() else []
    rows.append({"folder":folder,"label":label,"expected":expected,"downloaded":len(files),
                 "missing":max(0,expected-len(files)),"coverage_percent":round(100*len(files)/expected,2)})
inventory=pd.DataFrame(rows)
display(inventory)
if inventory.groupby("label").downloaded.sum().gt(0).sum()!=2:
    raise RuntimeError("Both Healthy and Unhealthy images are required.")
inventory.to_csv(ARTIFACTS/'01_inventory.csv',index=False)
report={"source":"IEEE DataPort","doi":"10.21227/a4f6-4413","source_url":"https://ieee-dataport.org/open-access/leaves-indias-most-famous-basil-plant-leaves-quality-dataset","expected":int(inventory.expected.sum()),"downloaded":int(inventory.downloaded.sum()),"missing":int(inventory.missing.sum()),"partial_dataset":bool((inventory.missing>0).any()),"label_rule":"Only explicitly labelled Healthy and Unhealthy folders are used."}
(ARTIFACTS/'01_collection_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print(report)

,folder,label,expected,downloaded,missing,coverage_percent
0,Amravati_Region_Basil_Plant_Healthy,Healthy,31,31,0,100.00
1,Nagpur_Region_Basil_Plant_Healthy,Healthy,473,428,45,90.49
2,Pune_Region_Basil_Plant_Healthy,Healthy,146,0,146,0.00
3,Basil_Plant_Unhealthy,Unhealthy,481,439,42,91.27


{'source': 'IEEE DataPort', 'doi': '10.21227/a4f6-4413', 'source_url': 'https://ieee-dataport.org/open-access/leaves-indias-most-famous-basil-plant-leaves-quality-dataset', 'expected': 1131, 'downloaded': 898, 'missing': 233, 'partial_dataset': True, 'label_rule': 'Only explicitly labelled Healthy and Unhealthy folders are used.'}


## Handoff to Person 2
Explain which folders are present, which are incomplete, and why this is a partial-dataset experiment. Person 2 must use the saved inventory and keep this limitation in the final report.